# 02 · Image detection

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/02_image_detection.ipynb)

Run a detector over a folder, read the results as a table, compare thresholds,
and plot the boxes yourself.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
fetch("demo_data_det.zip")
images = DATA / "data/data_example/images"

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["MDV6-yolov10-c"]:
    spe.download_model(model_id)

`detect` accepts a file, a folder, or a list of files. Results come back in input order,
and each one has a `.path`. Files that cannot be read are skipped with a warning, so always
match results by `r.path`, not by position.

In [ ]:
def progress(done, total, path):
    if done == total or done % 5 == 0:
        print(f"{done}/{total}")

results = spe.detect(images, model="MDV6-yolov10-c", threshold=0.1,
                     recursive=True, progress_callback=progress)

In [ ]:
import pandas as pd

rows = [
    {"file": Path(r.path).name, "label": d.label, "confidence": d.confidence,
     "x_min": d.bbox.x_min, "y_min": d.bbox.y_min, "x_max": d.bbox.x_max, "y_max": d.bbox.y_max}
    for r in results for d in r.detections
]
det = pd.DataFrame(rows)
det.sort_values("confidence", ascending=False).head(10)

Boxes are normalized to `[0, 1]`. Compare how many detections survive at different thresholds:

In [ ]:
pd.DataFrame(
    {t: {"detections": int((det.confidence >= t).sum()),
         "images with detections": det[det.confidence >= t].file.nunique()}
     for t in (0.1, 0.3, 0.5, 0.7)}
).T

## Plot boxes yourself

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image as PILImage

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, r in zip(axes, results[:3]):
    img = PILImage.open(r.path)
    w, h = img.size
    ax.imshow(img)
    for d in r.detections:
        if d.confidence < 0.3:
            continue
        b = d.bbox
        ax.add_patch(Rectangle((b.x_min * w, b.y_min * h), (b.x_max - b.x_min) * w,
                               (b.y_max - b.y_min) * h, fill=False, color="lime", lw=2))
        ax.text(b.x_min * w, b.y_min * h - 5, f"{d.label} {d.confidence:.2f}", color="lime")
    ax.set_title(Path(r.path).name[-24:])
    ax.axis("off")
plt.tight_layout()

## Try another detector

```python
spe.download_model("MDV6-yolov10-e")      # larger, slower, more accurate
spe.download_model("MDV6-mit-yolov9-c")   # MIT licence
spe.download_model("MDV6-apa-rtdetr-c")   # Apache-2.0 licence
```

Then pass its id as `model=`.